## Marco Lopez CS6220 HW 4

## Part 1: Frequent Itemsets

In [1]:
# Define transactions
transactions = [
["Bread", "Butter", "Milk"],
["Bread", "Butter", "Eggs"],
["Milk", "Cereal"],
["Bread", "Eggs"],
["Milk", "Cereal", "Yogurt"],
["Diapers", "Beer"],
["Diapers", "Beer", "Chips"],
["Apples", "Bananas"],
["Apples", "Bananas", "Milk"],
["Bread", "Butter", "Milk", "Eggs"],
["Cereal", "Milk"],
["Bread", "Butter", "Milk", "Eggs"],
["Apples", "Milk"],
["Diapers", "Beer", "Bread", "Eggs"],
["Yogurt", "Cereal"],
["Bread", "Butter", "Bananas", "Milk"],
["Milk", "Eggs"],
["Apples", "Bananas", "Yogurt"],
["Bread", "Butter", "Cereal", "Milk"],
["Chips", "Salsa"]
]

In [2]:
# Implement support function
# Percentage of total transactions that contain the itemset
def support(candidate_itemset, transactions):
    count = 0

    for transaction in transactions:
        if set(candidate_itemset).issubset(set(transaction)):
            count += 1

    return count / len(transactions)

In [3]:
# Implement confidence function
# Measures conditional probability of association rule
# Tells you how likely it is to find item Y given your transactions also has item X
def confidence(antecedent, consequent, transactions):
    combined = antecedent + consequent

    return support(combined, transactions) / support(antecedent, transactions)

In [4]:
# Implement Apriori function
# Core purpose: find frequent itemsets
from itertools import combinations

def apriori(min_sup, transactions):
    # Get all unique items
    items = sorted(set(item for transaction in transactions for item in transaction))

    # Generate 1-itemsets
    candidates = [frozenset([item]) for item in items]

    # Count and prune 1-itemsets
    frequent = {
        itemset for itemset in candidates
        if support(itemset, transactions) >= min_sup
    }
    
    # Store the frequent 1-item sets
    frequent_itemsets = {1: frequent}

    # Start at 2 to generate 2-item sets, later increment k
    k = 2

    # Repeat until no more frequent itemsets
    while frequent:

        # JOIN: Combine (k-1) itemsets to create k-itemset candidates
        candidates = set()

        frequent_list = list(frequent)

        for i in range(len(frequent_list)):
            for j in range(i + 1, len(frequent_list)):

                candidate = frequent_list[i] | frequent_list[j]

                if len(candidate) == k:
                    candidates.add(candidate)

        # PRUNE: Prune any candidates with infrequent k-1 itemsets
        pruned_candidates = set()

        for candidate in candidates:

            subsets = combinations(candidate, k - 1)

            if all(frozenset(subset) in frequent for subset in subsets):
                pruned_candidates.add(candidate)

        # COUNT: Calculate supports and remove any candiates below min_sup
        frequent = {
            candidate for candidate in pruned_candidates
            if support(candidate, transactions) >= min_sup
        }

        if frequent:
            frequent_itemsets[k] = frequent

        k += 1

    return frequent_itemsets

## Part 2: Association Rules

In [5]:
# Implement Apriori rules function, to find Association rules using Aprori and the Anti-monotonocity property

from itertools import combinations
def apriori_rules(min_conf, min_support, transactions):

    # PHASE 1: Generate the frequent itemsets using prior apriori function
    frequent_itemsets = apriori(min_support, transactions)
    # Store Association rules
    rules = []

    # PHASE 2: Generate the association rules
    # Loop through all frequent itemsets
    for k, itemsets in frequent_itemsets.items():

        # Skip if itemset only has one item, you need at least 2 to make association rule
        if k <= 1:
            continue
        
        # Loop through each frequent itemset
        for itemset in itemsets:

            # Generate 1-item consequents
            consequents = {frozenset([item]) for item in itemset}
            
            # Size of consequent
            m = 1
            
            # Continues while there are new consequents to evaluate
            while consequents:

                frequent_consequents = set()

                # COUNT: Calculate confidence
                for consequent in consequents:
                    antecedent = itemset - consequent

                    conf = confidence(antecedent, consequent, transactions)

                    # Add the association rule if it passes minimum confidence
                    if conf >= min_conf:
                        frequent_consequents.add(consequent)
                        rules.append((antecedent, consequent, conf))

                # JOIN: Generate larger consequents
                candidates = set()
                frequent_list = list(frequent_consequents)

                for i in range(len(frequent_list)):
                    for j in range(i + 1, len(frequent_list)):

                        candidate = frequent_list[i] | frequent_list[j]

                        if len(candidate) == m + 1:
                            candidates.add(candidate)

                # PRUNE
                pruned_candidates = set()

                for candidate in candidates:

                    subsets = combinations(candidate, m)
                    
                    # Subset must be a frequent consequent itself to be included
                    if all(frozenset(subset) in frequent_consequents
                           for subset in subsets):
                        pruned_candidates.add(candidate)

                m += 1

                consequents = pruned_candidates
                
                # No more frequent item sets
                if m >= k:
                    break

    # Returns the final association rules         
    return rules

## Part 3: Hierarchical Clustering

In [6]:
data = open("clustering.txt").read()

In [7]:
# Define cluster distance functions
def min_dist(d1, d2):
    return min(d1, d2)

def max_dist(d1, d2):
    return max(d1, d2)

In [8]:
# Import libraries
import numpy as np
import heapq

# Implement hierarchical clustering
def hierarchical_clustering(data, cluster_dist):
    # Number of datapoints
    n = len(data)

    # Initialize each datapoint as its own cluster
    clusters = set(range(n))

    # Stores the distance between every pair of datapoints
    distances = {}

    # Priority queue for finding the nearest clusters
    heap = []

    # Calculate initial pairwise distances for the datapoints
    for i in range(n):
        for j in range(i + 1, n):

            d = np.linalg.norm(data[i] - data[j])

            distances[(i, j)] = d
            heapq.heappush(heap, (d, i, j))

    # Store dendrogram
    dendrogram = []

    # New cluster IDs will start after the original datapoints
    next_id = n
    
    # While there are still multiple clusters that exist
    while len(clusters) > 1:
        # Find the nearest pair of active clusters using heap
        d, a, b = heapq.heappop(heap)

        # Skip if a or b are no longer a cluster
        if a not in clusters or b not in clusters:
            continue

        # Records the merge
        dendrogram.append((next_id, a, b, d))

        # Update distances from all clusters to the new cluster
        for c in clusters:

            if c != a and c != b:

                d1 = distances[tuple(sorted((a, c)))]
                d2 = distances[tuple(sorted((b, c)))]

                new_dist = cluster_dist(d1, d2)

                distances[tuple(sorted((next_id, c)))] = new_dist

                heapq.heappush(heap, (new_dist, next_id, c))

        # Remove old clusters and add the new merged cluster
        clusters.remove(a)
        clusters.remove(b)
        clusters.add(next_id)

        next_id += 1

    return dendrogram

## Report

